# 01 · Ingestion — IBM HR Employee Attrition

**Stage 1 of the Tier A pipeline** (`DOCS/STRUCTURE.md`). Acquire the raw file, log its lineage,
validate it against a written schema, and record what governance applies — *before* anything is
cleaned, dropped, or analysed.

**Business question this project answers** (Stage 0, locked in `IMPLEMENTATION_PLAN.md`):

> Among current employee profiles, which characteristics are genuinely associated with attrition —
> once features that are statistically indistinguishable from random noise are stripped out and pay
> is separated from seniority — and how well can the survivors score attrition risk *before* an
> engagement survey is run?

**What this notebook is looking for.** Not errors. This file is famously clean — 0 nulls, 0
duplicates. The interesting question at ingestion is the opposite one: *what does a file this clean
still fail to tell us?* Three things get raised here and resolved downstream:

1. Which columns carry **zero information** (and are any of them disguised as real measurements?)
2. Is the file **internally consistent** — do the tenure fields obey the arithmetic they must?
3. Is the sample a **complete population or a subset**, and if a subset, is the missing part random?

In [1]:
import hashlib
import json
import warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)

SRC_NAME = "WA_Fn-UseC_-HR-Employee-Attrition.csv"
ROOT = Path.cwd() if (Path.cwd() / SRC_NAME).exists() else Path.cwd().parent
RAW = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
REPORTS = ROOT / "reports"
for d in (RAW, PROC, REPORTS):
    d.mkdir(parents=True, exist_ok=True)

print("project root :", ROOT.name)
print("source file  :", SRC_NAME)

project root : IBM HR Analytics
source file  : WA_Fn-UseC_-HR-Employee-Attrition.csv


## 1.1 Load and log lineage

Immutability rule (STRUCTURE.md §1): the raw file is copied once, hashed, and never edited again.
Every downstream artefact traces back to this hash.

In [2]:
src_path = ROOT / SRC_NAME
raw_bytes = src_path.read_bytes()
sha256 = hashlib.sha256(raw_bytes).hexdigest()

df = pd.read_csv(src_path)

lineage = pd.DataFrame([{
    "artefact": "raw_source",
    "source_file": SRC_NAME,
    "publisher": "IBM Data Scientists (synthetic demo dataset)",
    "sha256": sha256,
    "bytes": len(raw_bytes),
    "pulled_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "n_rows": df.shape[0],
    "n_cols": df.shape[1],
    "delimiter": ",",
    "header": "row 0",
    "encoding": "utf-8",
}])
lineage.to_csv(RAW / "_lineage.csv", index=False)

print(f"shape        : {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"sha256       : {sha256}")
print(f"dtypes       : {dict(df.dtypes.astype(str).value_counts())}")
print(f"memory       : {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

shape        : 1,470 rows x 35 columns
sha256       : a5c31e38bd7fafc9bc333884eb181b06b41b8e5e488e8f7ccb27199fb3be7659
dtypes       : {'int64': np.int64(26), 'str': np.int64(9)}
memory       : 0.50 MB


In [3]:
df.head(3).T.head(12)

,0,1,2
Age,41,49,37
Attrition,Yes,No,Yes
BusinessTravel,Travel_Rarely,Travel_Frequently,Travel_Rarely
DailyRate,1102,279,1373
Department,Sales,Research & Development,Research & Development
DistanceFromHome,1,8,2
Education,2,1,2
EducationField,Life Sciences,Life Sciences,Other
EmployeeCount,1,1,1
EmployeeNumber,1,2,4


## 1.2 Schema validation

One row per rule, pass/fail, written to `data/raw/_schema_validation.csv`. Three families of rule:

- **Domain** — is every value in the documented set of allowed values?
- **Plausibility** — is every value in a physically sensible range?
- **Internal consistency** — do the tenure fields obey the arithmetic they logically must?

The third family is the one that earns its keep on a file with no nulls.

In [4]:
LIKERT = ["EnvironmentSatisfaction", "JobSatisfaction", "RelationshipSatisfaction",
          "WorkLifeBalance", "JobInvolvement"]

domain_rules = {
    "Attrition": {"Yes", "No"},
    "Gender": {"Male", "Female"},
    "OverTime": {"Yes", "No"},
    "Over18": {"Y"},
    "BusinessTravel": {"Non-Travel", "Travel_Rarely", "Travel_Frequently"},
    "Department": {"Sales", "Research & Development", "Human Resources"},
    "MaritalStatus": {"Single", "Married", "Divorced"},
    "Education": set(range(1, 6)),
    "JobLevel": set(range(1, 6)),
    "StockOptionLevel": set(range(0, 4)),
    "PerformanceRating": set(range(1, 5)),
    "TrainingTimesLastYear": set(range(0, 7)),
    **{c: {1, 2, 3, 4} for c in LIKERT},
}

range_rules = {
    "Age": (18, 60), "MonthlyIncome": (1_000, 20_000), "DistanceFromHome": (1, 29),
    "PercentSalaryHike": (11, 25), "DailyRate": (100, 1500), "HourlyRate": (30, 100),
    "MonthlyRate": (2000, 27000), "TotalWorkingYears": (0, 40), "YearsAtCompany": (0, 40),
    "NumCompaniesWorked": (0, 9),
}

# Internal-consistency rules: (label, boolean Series that must be all-True)
consistency_rules = {
    "YearsAtCompany <= TotalWorkingYears": df.YearsAtCompany <= df.TotalWorkingYears,
    "YearsInCurrentRole <= YearsAtCompany": df.YearsInCurrentRole <= df.YearsAtCompany,
    "YearsWithCurrManager <= YearsAtCompany": df.YearsWithCurrManager <= df.YearsAtCompany,
    "YearsSinceLastPromotion <= YearsAtCompany": df.YearsSinceLastPromotion <= df.YearsAtCompany,
    "Age - TotalWorkingYears >= 16": (df.Age - df.TotalWorkingYears) >= 16,
}

rows = []
for col, allowed in domain_rules.items():
    bad = (~df[col].isin(allowed)).sum()
    rows.append({"family": "domain", "rule": f"{col} in {sorted(allowed)[:5]}{'...' if len(allowed) > 5 else ''}",
                 "column": col, "violations": int(bad), "status": "PASS" if bad == 0 else "FAIL"})
for col, (lo, hi) in range_rules.items():
    bad = ((df[col] < lo) | (df[col] > hi)).sum()
    rows.append({"family": "plausibility", "rule": f"{lo} <= {col} <= {hi}",
                 "column": col, "violations": int(bad), "status": "PASS" if bad == 0 else "FAIL"})
for label, ok in consistency_rules.items():
    bad = int((~ok).sum())
    rows.append({"family": "consistency", "rule": label, "column": "-",
                 "violations": bad, "status": "PASS" if bad == 0 else "FAIL"})

schema = pd.DataFrame(rows)
schema.to_csv(RAW / "_schema_validation.csv", index=False)

print(schema.groupby(["family", "status"]).size().to_string())
print()
print("FAILURES:", (schema.status == "FAIL").sum())
schema[schema.status == "FAIL"]

family        status
consistency   PASS       5
domain        PASS      17
plausibility  PASS      10

FAILURES: 0


,family,rule,column,violations,status


### Finding — the file is internally consistent by construction

All five consistency rules pass with zero violations. `min(Age − TotalWorkingYears)` is well above
the legal-working floor. No employee has been at the company longer than they have worked, held
their role longer than they have been employed, or reported to a manager for longer than they have
existed at the firm.

**So What:** a generator that respected tenure arithmetic produced this file. That matters later,
because it makes the *inconsistency* we find in §1.3 — three columns of pure noise sitting inside a
carefully-constrained schema — a deliberate-looking artefact rather than sloppiness. It also means
zero cleaning effort is available to spend here, so the cleaning notebook is a **forensics**
notebook instead.

In [5]:
print("min(Age - TotalWorkingYears) :", int((df.Age - df.TotalWorkingYears).min()))
print("nulls, whole file            :", int(df.isnull().sum().sum()))
print("duplicate rows               :", int(df.duplicated().sum()))
print("duplicate EmployeeNumber     :", int(df.EmployeeNumber.duplicated().sum()))
print()
print("Grain confirmed: one row = one employee snapshot.")

min(Age - TotalWorkingYears) : 18
nulls, whole file            : 0
duplicate rows               : 0
duplicate EmployeeNumber     : 0

Grain confirmed: one row = one employee snapshot.


> **Zero nulls is not a quality signal — it is a synthetic-data artefact.** A real HRIS export
> carries missing hire dates, blank manager IDs, and free-text role names. `DOCS` §2 says the same.
> The absence of cleaning work here is precisely why this file cannot stand in for a production
> pipeline, and the report says so.

## 1.3 Information content — which columns are inert?

Two distinct failure modes, and only the first is documented in the data assessment:

1. **Zero variance** — one value for all 1,470 rows. Trivially detectable.
2. **Variance without information** — plenty of distinct values, no relationship to anything.
   Undetectable by a profiler, and *far* more dangerous, because it looks like a real measurement.

In [6]:
nunique = df.nunique()
zero_var = nunique[nunique == 1]
print("Zero-variance columns (drop in Stage 2):")
for c in zero_var.index:
    print(f"  {c:<18} constant value = {df[c].iloc[0]!r}")

print("\nNear-zero-variance (retained, but flagged):")
for c in df.columns:
    vc = df[c].value_counts(normalize=True)
    if 1 < df[c].nunique() <= 5 and vc.iloc[0] >= 0.80:
        print(f"  {c:<18} {vc.index[0]!r} covers {vc.iloc[0]:.1%} of rows")

Zero-variance columns (drop in Stage 2):
  EmployeeCount      constant value = np.int64(1)
  Over18             constant value = 'Y'
  StandardHours      constant value = np.int64(80)

Near-zero-variance (retained, but flagged):
  Attrition          'No' covers 83.9% of rows
  PerformanceRating  np.int64(3) covers 84.6% of rows


### The rate columns: variance without information

`DailyRate`, `HourlyRate` and `MonthlyRate` are presented as compensation fields. If they were, they
would be arithmetically tied to each other and to `MonthlyIncome` — and to `JobLevel`, since pay
tracks seniority in every real organisation.

The check below is the first contact with what becomes this project's central finding. It is raised
here and **convicted with evidence in `02_cleaning.ipynb` §2.1**.

In [7]:
rate_cols = ["DailyRate", "HourlyRate", "MonthlyRate"]
anchors = ["MonthlyIncome", "JobLevel", "TotalWorkingYears", "Age"]

spear = df[rate_cols + anchors].corr(method="spearman").loc[rate_cols, anchors]
print("Spearman rho of the three 'rate' columns against real pay/seniority anchors:")
print(spear.round(3).to_string())
print()
print("For contrast, MonthlyIncome vs the same anchors:")
print(df[anchors].corr(method="spearman").loc[["MonthlyIncome"], anchors].round(3).to_string())
print()
print(f"max |rho| anywhere in the rate block : {spear.abs().to_numpy().max():.3f}")

Spearman rho of the three 'rate' columns against real pay/seniority anchors:
             MonthlyIncome  JobLevel  TotalWorkingYears    Age
DailyRate            0.016     0.004              0.021  0.007
HourlyRate          -0.020    -0.034             -0.012  0.029
MonthlyRate          0.054     0.053              0.013  0.017

For contrast, MonthlyIncome vs the same anchors:
               MonthlyIncome  JobLevel  TotalWorkingYears    Age
MonthlyIncome            1.0      0.92               0.71  0.472

max |rho| anywhere in the rate block : 0.054


**Raised, not yet resolved.** Three columns that look like pay data correlate with real pay at
|ρ| < 0.06 — while `MonthlyIncome` itself correlates with `JobLevel` at ρ = 0.92. Whatever these
three columns are, they are not this employee's pay.

They are **not dropped**. Stage 2 establishes what they actually are, and Stage 5a puts them to
work as a calibrated noise floor.

## 1.4 Sampling frame — is this the whole population?

`EmployeeNumber` runs from 1 upward but the file has fewer rows than its maximum ID. If the missing
IDs are employees who already left, the dataset is a survivorship-biased sample and the attrition
rate is understated. This is testable, so we test it rather than assume.

In [8]:
from scipy import stats

en = df.EmployeeNumber.to_numpy()
y = (df.Attrition == "Yes").astype(int).to_numpy()

print(f"EmployeeNumber range   : {en.min()} .. {en.max()}")
print(f"rows present           : {len(en):,}")
print(f"IDs absent             : {en.max() - len(en):,}  ({1 - len(en) / en.max():.1%} of the ID space)")
print(f"strictly increasing    : {bool(pd.Series(en).is_monotonic_increasing)}")

# Test 1 - is an employee followed by an ID gap more likely to have left?
gap_after = np.diff(np.append(en, en.max() + 1)) - 1
ct = pd.crosstab(gap_after > 0, y)
chi2, p_gap, _, _ = stats.chi2_contingency(ct)
print(f"\nattrition | followed by gap : {y[gap_after > 0].mean():.3f}  (n={int((gap_after > 0).sum()):,})")
print(f"attrition | no gap follows  : {y[gap_after == 0].mean():.3f}  (n={int((gap_after == 0).sum()):,})")
print(f"chi-square p                : {p_gap:.3f}")

# Test 2 - does attrition drift across the ID space?
dec = pd.qcut(df.EmployeeNumber, 10, labels=False)
by_dec = pd.crosstab(dec, y, normalize="index")[1]
r_trend, p_trend = stats.spearmanr(dec, y)
print(f"\nattrition rate by ID decile : {np.round(by_dec.to_numpy(), 3)}")
print(f"Spearman(ID decile, attrition) rho = {r_trend:+.3f}, p = {p_trend:.3f}")

EmployeeNumber range   : 1 .. 2068
rows present           : 1,470
IDs absent             : 598  (28.9% of the ID space)
strictly increasing    : True

attrition | followed by gap : 0.155  (n=444)
attrition | no gap follows  : 0.164  (n=1,026)
chi-square p                : 0.748

attrition rate by ID decile : [0.177 0.143 0.15  0.143 0.197 0.204 0.156 0.15  0.163 0.129]
Spearman(ID decile, attrition) rho = -0.011, p = 0.666


### Finding (null result, reported not buried)

No detectable relationship between an employee's position in the ID space and whether they left
(χ² p = 0.75; rank correlation p ≈ 0.6). The 598 absent IDs are **not** a hidden leaver population.

**So What:** the 16.1% attrition rate can be read at face value for this cohort, and no
survivorship correction is needed. This is a check that was run and passed — recorded here so that
a reviewer does not have to wonder whether anyone looked.

## 1.5 Target

`Attrition` is the label. Its balance sets the metric policy for the entire project.

In [9]:
tgt = df.Attrition.value_counts()
rate = (df.Attrition == "Yes").mean()
print(tgt.to_string())
print(f"\nattrition rate           : {rate:.4f}  ({tgt['Yes']} of {len(df):,})")
print(f"imbalance ratio          : 1 : {tgt['No'] / tgt['Yes']:.1f}")
print(f"majority-class accuracy  : {1 - rate:.1%}   <-- a model that predicts 'stay' for everyone")
print(f"...and its recall on leavers: 0.0%")

Attrition
No     1233
Yes     237

attrition rate           : 0.1612  (237 of 1,470)
imbalance ratio          : 1 : 5.2
majority-class accuracy  : 83.9%   <-- a model that predicts 'stay' for everyone
...and its recall on leavers: 0.0%


> **Metric policy, fixed now so it cannot be chosen later to flatter a result:** accuracy is
> disqualified. A model that predicts "stays" for all 1,470 employees scores 83.9% accuracy and
> catches zero leavers. The primary metrics for this project are **PR-AUC and recall on the
> attrition class**, with ROC-AUC reported for comparability. (`DOCS` §6.)

## 1.6 Governance

Synthetic data, so no PDP/GDPR exposure — but the *modelling* exposure is real, because the columns
below are protected or protected-adjacent attributes and the downstream artefact is a score that
decides who gets a retention conversation.

In [10]:
gov = pd.DataFrame([
    {"item": "Data classification", "value": "Public synthetic demo data (IBM Watson Analytics sample)"},
    {"item": "Real persons", "value": "No - fictional employees, no PII, no re-identification risk"},
    {"item": "Safe to commit", "value": "Yes (though the portfolio .gitignore ships notebooks + report only)"},
    {"item": "Protected attributes present", "value": "Gender, Age"},
    {"item": "Protected-adjacent attributes", "value": "MaritalStatus, EducationField"},
    {"item": "Downstream obligation", "value": "Fairness audit is MANDATORY (STRUCTURE.md Cross-Cutting) - 05_modeling S6.5"},
    {"item": "Real-world caveat", "value": "An equivalent Indonesian HR export would require UU PDP handling before analysis"},
    {"item": "Purpose limitation", "value": "Analysis + portfolio demonstration; no deployment claim is made"},
])
gov.to_csv(RAW / "_governance.csv", index=False)
print(gov.to_string(index=False))

print("\nSubgroup counts that the fairness audit will have to work with:")
for c in ["Gender", "MaritalStatus"]:
    print(f"\n{c}:")
    print(pd.concat([df[c].value_counts().rename("n"),
                     df.groupby(c).Attrition.apply(lambda s: (s == "Yes").mean()).round(3).rename("attrition_rate")],
                    axis=1).to_string())

                         item                                                                            value
          Data classification                         Public synthetic demo data (IBM Watson Analytics sample)
                 Real persons                      No - fictional employees, no PII, no re-identification risk
               Safe to commit              Yes (though the portfolio .gitignore ships notebooks + report only)
 Protected attributes present                                                                      Gender, Age
Protected-adjacent attributes                                                    MaritalStatus, EducationField
        Downstream obligation      Fairness audit is MANDATORY (STRUCTURE.md Cross-Cutting) - 05_modeling S6.5
            Real-world caveat An equivalent Indonesian HR export would require UU PDP handling before analysis
           Purpose limitation                  Analysis + portfolio demonstration; no deployment claim is made



## 1.7 Persist the immutable copy

In [11]:
df.to_csv(RAW / SRC_NAME, index=False)
print("written:", (RAW / SRC_NAME).relative_to(ROOT))
for f in sorted(RAW.glob("_*.csv")):
    print("written:", f.relative_to(ROOT))
print("\nThis copy is never edited again. Stage 2 reads it and writes to data/processed/.")

written: data\raw\WA_Fn-UseC_-HR-Employee-Attrition.csv
written: data\raw\_governance.csv
written: data\raw\_lineage.csv
written: data\raw\_schema_validation.csv

This copy is never edited again. Stage 2 reads it and writes to data/processed/.


---

## Stage 1 — Gate Checklist (`DOCS/STRUCTURE.md`)

- [x] **Raw data saved to `data/raw/`** (untouched copy, SHA-256 `…` logged)
- [x] **Source, timestamp, and row/column counts logged** → `_lineage.csv`
- [x] **Schema matches expectations** — 32 rules across 3 families, **0 failures**;
      internal-consistency rules pass, which is itself a finding (synthetic generator)
- [x] **PII / data governance review completed** → `_governance.csv`; no PII, but `Gender`,
      `Age` and `MaritalStatus` trigger a mandatory fairness audit downstream

### Carried forward to Stage 2

| # | Raised here | Resolved in |
|---|---|---|
| 1 | `EmployeeCount`, `StandardHours`, `Over18` are zero-variance | `02_cleaning` §2.2 — dropped |
| 2 | `PerformanceRating` is 84.6% one value | `02_cleaning` §2.2 — retained, null verdict expected |
| 3 | `DailyRate`/`HourlyRate`/`MonthlyRate` show no link to pay or seniority | `02_cleaning` §2.1 — convicted, then repurposed as a placebo control |
| 4 | `MonthlyIncome` ↔ `JobLevel` at ρ = 0.92 | `02_cleaning` §2.4 — collinearity register |
| 5 | Sampling-frame check | **closed** — null result, no survivorship bias |

**Next:** `02_cleaning.ipynb`.